# Record a solve, then ask *what if*

In [ ]:
import runpy, warnings
warnings.simplefilter("ignore")
from difflow import debug
from difflow.debug.record import compare, state_at, summarize

fs = runpy.run_path("plant.py")["fs"]
trace = debug.record(fs, {"acceleration": "wegstein"})
for row in summarize(trace)["iterations"]:
    print(f"frame {row['frame']:3d}  iter {row['iteration']}  {row['residual']:.2e}  {row['largest']}")

In [ ]:
from twostates import bistable          # a loop with steady states at x = 0 and x = 1

kw = {"use_initialization": False, "acceleration": "wegstein"}
base = debug.record(bistable(), kw)
k = summarize(base)["iterations"][0]["frame"]

replay = debug.record(bistable(), kw, baseline=base["frames"],
                      edits=[{"frame": k, "stream": "tear", "field": "F_A", "value": 0.8}])
compare(base, replay)

In [ ]:
import matplotlib.pyplot as plt

for label, t in [("recorded", base), ("what-if: tear = 0.8", replay)]:
    res = [f["residual"] for f in t["frames"] if f["checkpoint"] == "iteration"]
    x = float(state_at(t, len(t["frames"]) - 1)["out"]["F_A"])
    plt.semilogy([max(r, 1e-17) for r in res], "o-", label=f"{label} -> x = {x:.2g}")
plt.xlabel("tear iteration"); plt.ylabel("residual"); plt.legend();